# 73 - Experimento MPNet + DBSCAN

Este notebook compara embeddings `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` com DBSCAN para agrupar descricoes normalizadas de itens de supermercado. Compara duas sementes de amostragem, 42 e 123, usando a mesma base e periodo do notebook 72. Nao usa NCM nem rotulos humanos; ruido e tratado como resultado explicito.

Execute de cima para baixo em uma sessao limpa. Cada execucao grava relatorios em uma pasta UTC nova dentro de `reports/mpnet_dbscan/`. O primeiro uso do MPNet pode baixar o modelo do Hugging Face.


In [14]:
from datetime import datetime, timezone
from pathlib import Path
import json
import logging

import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn import config_context

from nofis_classifier.schema import DESCRIPTION_NORMALIZED, FREQUENCY

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook73")
pd.set_option("display.max_colwidth", 140)


## 1. Parametros

`eps` e o raio maximo de vizinhanca em distancia cosseno; `min_samples` e o minimo de pontos para formar uma regiao densa. Silhouette e calculada somente nos itens nao classificados como ruido. Frequencia pondera o ajuste, enquanto cada descricao unica continua sendo um ponto. A grade e as amostras sao exploratorias; resultados devem ser lidos junto da quantidade de clusters e da fracao de ruido.

O experimento repete a grade com as sementes 42 e 123 para avaliar a sensibilidade a amostragem. MPNet gera embeddings com o tokenizer e a representacao definidos pelo modelo; nao ha um parametro `analyzer` equivalente ao do TF-IDF. Para comparar analyzers, use o notebook 72.


In [15]:
DATASET_NAME = "2025_NCMs-sem-duplicatas"  # Nome da pasta em data/processed; altere apenas este valor para trocar a base.

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Nao encontrei a raiz do repositorio com data/processed/.")
DATASET_DIR = PROJECT_ROOT / "data" / "processed" / DATASET_NAME
DATASET_FILES = sorted(DATASET_DIR.glob("*.parquet"))
if not DATASET_FILES:
    raise FileNotFoundError(f"Nenhum Parquet encontrado para {DATASET_NAME}: {DATASET_DIR}")
if len(DATASET_FILES) > 1:
    raise ValueError(
        f"Mais de um Parquet encontrado em {DATASET_DIR}; selecione o arquivo/periodo explicitamente: "
        f"{[path.name for path in DATASET_FILES]}"
    )
DATASET_PATH = DATASET_FILES[0]
if not DATASET_PATH.stem.startswith(f"{DATASET_NAME}-"):
    raise ValueError(
        f"O nome do arquivo deve comecar com '{DATASET_NAME}-' para derivar o periodo: "
        f"{DATASET_PATH.name}"
    )
PERIOD_LABEL = DATASET_PATH.stem[len(DATASET_NAME) + 1:]
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
MAX_DESCRIPTIONS = None  # Limite inicial igual ao notebook 72 para controlar o custo do DBSCAN.
SAMPLE_SEEDS = [42]
ENCODE_BATCH_SIZE = 64
DBSCAN_N_JOBS = 1
PAIRWISE_WORKING_MEMORY_MIB = 64
EPS_VALUES = [0.01, 0.02, 0.03, 0.04, 0.05]
MIN_SAMPLES_VALUES = [10, 20, 30, 100]
OUTPUT_ROOT = Path("reports/mpnet_dbscan")


## 2. Fun??es da an?lise

A prepara??o consolida duplicatas sem alterar a frequ?ncia total. O resumo mant?m o r?tulo `-1` para ru?do. A silhouette exclui ru?do e s? ? calculada quando h? pelo menos dois clusters e mais de um item em cada parti??o considerada.


In [16]:
def prepare_items(frame):
    """Limpa descri??es, agrega duplicatas e preserva a frequ?ncia total."""
    initial_frequency = frame[FREQUENCY].sum()
    prepared = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].copy()
    prepared[DESCRIPTION_NORMALIZED] = prepared[DESCRIPTION_NORMALIZED].astype("string").str.strip()
    prepared = prepared[prepared[DESCRIPTION_NORMALIZED].notna() & prepared[DESCRIPTION_NORMALIZED].ne("")]
    prepared = prepared[prepared[FREQUENCY] >= 1].copy()
    prepared = prepared.groupby(DESCRIPTION_NORMALIZED, as_index=False, sort=False)[FREQUENCY].sum()
    if prepared[FREQUENCY].sum() != initial_frequency:
        raise ValueError("A prepara??o alterou a soma de frequ?ncia.")
    return prepared.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)


def score_configuration(vectors, labels):
    """Calcula m?tricas de DBSCAN e silhouette sem incluir pontos de ru?do."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    clustered_labels = labels[~noise_mask]
    cluster_count = len(np.unique(clustered_labels))
    score = np.nan
    if 2 <= cluster_count < len(clustered_labels):
        score = float(silhouette_score(vectors[~noise_mask], clustered_labels, metric="cosine"))
    return {
        "cluster_count_excluding_noise": cluster_count,
        "noise_count": int(noise_mask.sum()),
        "noise_fraction": float(noise_mask.mean()),
        "silhouette_cosine_without_noise": score,
    }


def summarize_clusters(items, labels):
    """Retorna atribui??es e resumo dos clusters, incluindo a linha de ru?do."""
    result = items.copy()
    result["cluster"] = labels
    result["ruido"] = result["cluster"].eq(-1)
    summary = (
        result.groupby("cluster", as_index=False)
        .agg(descricoes_unicas=(DESCRIPTION_NORMALIZED, "size"), frequencia_total=(FREQUENCY, "sum"))
        .sort_values("descricoes_unicas", ascending=False)
    )
    return result, summary


## 3. Carregar, preparar e gerar embeddings

A amostra e selecionada separadamente para cada semente. O modelo MPNet transforma cada amostra em embeddings normalizados, que serao reutilizados em toda a grade de DBSCAN daquela semente.


In [17]:
raw_items = pd.read_parquet(DATASET_PATH)
required_columns = {DESCRIPTION_NORMALIZED, FREQUENCY}
missing_columns = required_columns - set(raw_items.columns)
if missing_columns:
    raise ValueError(f"Colunas ausentes no dataset: {sorted(missing_columns)}")
base_items = prepare_items(raw_items)
logger.info("Descricoes disponiveis apos preparacao: %d", len(base_items))
logger.info("Carregando modelo %s", MODEL_NAME)
encoder = SentenceTransformer(MODEL_NAME)

items_by_seed = {}
vectors_by_seed = {}
for sample_seed in SAMPLE_SEEDS:
    sampled_items = base_items
    if MAX_DESCRIPTIONS is not None and len(base_items) > MAX_DESCRIPTIONS:
        sampled_items = base_items.sample(n=MAX_DESCRIPTIONS, random_state=sample_seed)
    sampled_items = sampled_items.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
    vectors = encoder.encode(
        sampled_items[DESCRIPTION_NORMALIZED].astype(str).tolist(),
        batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    items_by_seed[sample_seed] = sampled_items
    vectors_by_seed[sample_seed] = vectors
    logger.info(
        "Seed %d: %d descricoes, embeddings %s, frequencia total %d",
        sample_seed, len(sampled_items), vectors.shape, sampled_items[FREQUENCY].sum(),
    )

display(pd.DataFrame([
    {
        "sample_seed": sample_seed,
        "descricoes": len(items_by_seed[sample_seed]),
        "dimensoes_embedding": vectors_by_seed[sample_seed].shape[1],
        "frequencia_total": int(items_by_seed[sample_seed][FREQUENCY].sum()),
    }
    for sample_seed in SAMPLE_SEEDS
]))


2026-10-13 08:02:14,858 INFO Descricoes disponiveis apos preparacao: 257879
2026-10-13 08:02:14,858 INFO Carregando modelo sentence-transformers/paraphrase-multilingual-mpnet-base-v2
2026-10-13 08:02:14,910 INFO No device provided, using cpu
2026-10-13 08:02:15,259 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-13 08:02:15,291 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/modules.json?%2Fsentence-transformers%2Fparaphrase-multilingual-mpnet-base-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22f7640f94e81bb7f4f04daf1668850b38763a13d9%22 "HTTP/1.1 200 OK"
2026-10-13 08:02:15,442 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 3

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-10-13 08:02:20,026 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2 "HTTP/1.1 200 OK"
2026-10-13 08:02:20,257 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-13 08:02:20,428 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-13 08:02:59,691 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2 "HTTP/1.1 200 OK"
2026-10-13 08:02:59,860 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-13 08:02:59,879 INFO HTTP Request: HEAD https:

Batches:   0%|          | 0/4030 [00:00<?, ?it/s]

2026-10-13 10:36:15,131 INFO Seed 42: 257879 descricoes, embeddings (257879, 768), frequencia total 1238688


,sample_seed,descricoes,dimensoes_embedding,frequencia_total
0,42,257879,768,1238688


## 4. Ajustar DBSCAN e comparar parametros

DBSCAN usa distancia cosseno nos vetores densos do MPNet. A frequencia entra como `sample_weight`; silhouette e fracao de ruido contam descricoes unicas. Os embeddings de cada semente sao reutilizados nos ajustes correspondentes.


In [18]:
records = []
labels_by_run = {}
items_by_run = {}
for sample_seed in SAMPLE_SEEDS:
    items = items_by_seed[sample_seed]
    vectors = vectors_by_seed[sample_seed]
    for eps in EPS_VALUES:
        for min_samples in MIN_SAMPLES_VALUES:
            experiment_id = f"seed{sample_seed}_eps{eps:g}_min_samples{min_samples}"
            logger.info("Ajustando DBSCAN %s", experiment_id)
            model = DBSCAN(
                eps=eps, min_samples=min_samples, metric="cosine",
                algorithm="brute", n_jobs=DBSCAN_N_JOBS,
            )
            with config_context(working_memory=PAIRWISE_WORKING_MEMORY_MIB):
                model.fit(vectors, sample_weight=items[FREQUENCY].to_numpy(dtype=float))
            labels = model.labels_
            metrics = score_configuration(vectors, labels)
            records.append({
                "run_id": experiment_id, "sample_seed": sample_seed,
                "eps": eps, "min_samples": min_samples,
                "frequency_weighted": True, **metrics,
            })
            labels_by_run[experiment_id] = labels
            items_by_run[experiment_id] = items
results = pd.DataFrame(records).sort_values(
    ["silhouette_cosine_without_noise", "noise_fraction"],
    ascending=[False, True], na_position="last",
).reset_index(drop=True)
display(results)


2026-10-13 10:36:15,639 INFO Ajustando DBSCAN seed42_eps0.01_min_samples10
2026-10-13 13:13:49,844 INFO Ajustando DBSCAN seed42_eps0.01_min_samples20
2026-10-13 16:32:50,793 INFO Ajustando DBSCAN seed42_eps0.01_min_samples30
2026-10-13 20:10:27,550 INFO Ajustando DBSCAN seed42_eps0.01_min_samples100
2026-10-14 00:30:49,687 INFO Ajustando DBSCAN seed42_eps0.02_min_samples10
2026-10-14 01:57:27,307 INFO Ajustando DBSCAN seed42_eps0.02_min_samples20
2026-10-14 03:19:12,203 INFO Ajustando DBSCAN seed42_eps0.02_min_samples30
2026-10-14 04:40:45,951 INFO Ajustando DBSCAN seed42_eps0.02_min_samples100
2026-10-14 06:02:15,663 INFO Ajustando DBSCAN seed42_eps0.03_min_samples10
2026-10-14 07:24:11,791 INFO Ajustando DBSCAN seed42_eps0.03_min_samples20
2026-10-14 08:51:02,527 INFO Ajustando DBSCAN seed42_eps0.03_min_samples30
2026-10-14 10:13:23,523 INFO Ajustando DBSCAN seed42_eps0.03_min_samples100
2026-10-14 11:36:36,673 INFO Ajustando DBSCAN seed42_eps0.04_min_samples10
2026-10-14 13:00:35,44

,run_id,sample_seed,eps,min_samples,frequency_weighted,cluster_count_excluding_noise,noise_count,noise_fraction,silhouette_cosine_without_noise
0,seed42_eps0.01_min_samples100,42,0.01,100,True,1384,253912,0.984617,0.655873
1,seed42_eps0.02_min_samples100,42,0.02,100,True,1439,250317,0.970676,0.647580
2,seed42_eps0.03_min_samples100,42,0.03,100,True,1435,245006,0.950081,0.569508
3,seed42_eps0.02_min_samples30,42,0.02,30,True,5130,240658,0.933221,0.539875
4,seed42_eps0.02_min_samples20,42,0.02,20,True,7938,234908,0.910923,0.502403
5,seed42_eps0.03_min_samples30,42,0.03,30,True,4877,231357,0.897153,0.478642
6,seed42_eps0.01_min_samples30,42,0.01,30,True,5216,247570,0.960024,0.477183
7,seed42_eps0.04_min_samples100,42,0.04,100,True,1385,237600,0.921362,0.472993
8,seed42_eps0.03_min_samples20,42,0.03,20,True,7469,224129,0.869125,0.449254
9,seed42_eps0.02_min_samples10,42,0.02,10,True,16441,220402,0.854672,0.436348


## 5. Revisar um resultado e salvar relat?rios

A sele??o autom?tica usa a maior silhouette dispon?vel; confira tamb?m quantos clusters foram formados e a fra??o de ru?do antes de interpretar. Todos os resultados da grade s?o salvos, e os arquivos da execu??o n?o sobrescrevem resultados anteriores.


In [19]:
SELECTED_RUN_ID = results.iloc[0]["run_id"]
selected = results.set_index("run_id").loc[SELECTED_RUN_ID]
selected_items = items_by_run[SELECTED_RUN_ID]
selected_labels = labels_by_run[SELECTED_RUN_ID]
assignments, cluster_summary = summarize_clusters(selected_items, selected_labels)
display(cluster_summary)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_dir = OUTPUT_ROOT / f"{PERIOD_LABEL}_{RUN_ID}"
output_dir.mkdir(parents=True, exist_ok=False)
results.to_csv(output_dir / "metrics.csv", index=False)
assignments.to_csv(output_dir / "assignments.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv(output_dir / "cluster_summary.csv", index=False, encoding="utf-8-sig")
run_metadata = {
    "dataset_name": DATASET_NAME, "period": PERIOD_LABEL,
    "dataset_path": str(DATASET_PATH), "max_descriptions": MAX_DESCRIPTIONS,
    "sample_seeds": SAMPLE_SEEDS, "model_name": MODEL_NAME,
    "embedding_normalized": True, "frequency_weighted": True,
    "selected_run_id": SELECTED_RUN_ID, "eps_values": EPS_VALUES,
    "min_samples_values": MIN_SAMPLES_VALUES,
}
(output_dir / "run.json").write_text(json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8")
logger.info("Relatorios salvos em %s", output_dir)


,cluster,descricoes_unicas,frequencia_total
0,-1,253912,802300
719,718,977,999
1315,1314,529,529
306,305,45,103
570,569,33,421
...,...,...,...
16,15,1,432
15,14,1,100
13,12,1,579
12,11,1,337


2026-10-14 22:50:50,415 INFO Relatorios salvos em reports\mpnet_dbscan\202501-202512_20261015T015049Z


### Interpreta??o

- `noise_fraction` alta indica que muitas descri??es ficaram fora de regi?es densas para os par?metros escolhidos.
- Silhouette deve ser interpretada junto da quantidade de clusters e da fra??o de ru?do; uma pontua??o alta pode refletir poucos pontos classificados.
- A compara??o com TF-IDF ? explorat?ria: os dois m?todos usam a mesma amostra, mas representam similaridade de formas diferentes.
- DBSCAN n?o fornece uma opera??o direta para atribuir novas descri??es a clusters ap?s o ajuste e n?o define categorias sem revis?o humana.
